# ClinicalTrials.gov API sample ingestion

Extracts up to 200 studies, normalises trial and child records, and checks identifier uniqueness and parent-child relationships. This sample is separate from the AACT dataset used in the dashboard.

Run from the notebooks directory after creating data/raw and installing pandas, requests and Jupyter/ipykernel. Execution retrieves live records, so results may differ from the archived sample. Regenerated CSVs are written to data/processed; data/development contains the archived sample.

The revised date parsing has not yet been validated against the original raw dates, and the revised notebook has not been rerun. Historical structural checks are summarised in [sources and validation](../docs/sources-and-validation.md).


In [ ]:
# ClinicalTrials.gov API Ingestion

import requests
import json
import hashlib
import pandas as pd

from pathlib import Path
from datetime import datetime

In [ ]:
# Define and validate project cohort

base_url = "https://clinicaltrials.gov/api/v2/studies"

project_query = (
    "AREA[StudyType]INTERVENTIONAL "
    "AND AREA[LeadSponsorClass]INDUSTRY "
    "AND (AREA[InterventionType]DRUG OR AREA[InterventionType]BIOLOGICAL) "
    "AND AREA[StartDate]RANGE[2015-01-01,MAX]"
)

params = {
    "query.term": project_query,
    "pageSize": 5,
    "format": "json",
    "countTotal": "true"
}

response = requests.get(
    base_url,
    params=params,
    timeout=30
)

response.raise_for_status()

filtered_data = response.json()

print("Status:", response.status_code)
print("Studies returned:", len(filtered_data["studies"]))
print("Total studies matching project scope:", filtered_data.get("totalCount"))

In [ ]:
# Controlled project extraction - 200 studies

params = {
    "query.term": project_query,
    "pageSize": 50,
    "format": "json"
}

all_studies = []
next_page_token = None
max_records = 200
pages_downloaded = 0

while len(all_studies) < max_records:

    print(f"Requesting page {pages_downloaded + 1}...")

    if next_page_token:
        params["pageToken"] = next_page_token
    else:
        params.pop("pageToken", None)

    response = requests.get(
        base_url,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    page_data = response.json()

    all_studies.extend(page_data.get("studies", []))

    pages_downloaded += 1

    print(f"Studies downloaded: {len(all_studies)}")

    next_page_token = page_data.get("nextPageToken")

    if not next_page_token:
        break

all_studies = all_studies[:max_records]

print("Extraction complete")
print("Pages downloaded:", pages_downloaded)
print("Final records:", len(all_studies))

In [ ]:
# Save raw development extract

raw_extract = {
    "studies": all_studies
}

raw_extract_path = Path("../data/raw/clinicaltrials_project_dev_200.json")

with open(raw_extract_path, "w", encoding="utf-8") as file:
    json.dump(raw_extract, file, indent=2)

print("Saved:", raw_extract_path)
print("File exists:", raw_extract_path.exists())

In [ ]:
# Create extraction manifest

import hashlib

# Calculate SHA-256 hash of the raw JSON file
sha256 = hashlib.sha256()

with open(raw_extract_path, "rb") as file:
    for chunk in iter(lambda: file.read(8192), b""):
        sha256.update(chunk)

file_hash = sha256.hexdigest()

manifest = {
    "source": "ClinicalTrials.gov API v2",
    "endpoint": base_url,
    "extract_timestamp": datetime.now().astimezone().isoformat(),
    "query": project_query,
    "page_size": 50,
    "pages_downloaded": pages_downloaded,
    "records_downloaded": len(all_studies),
    "development_limit": max_records,
    "raw_file": raw_extract_path.name,
    "sha256": file_hash
}

manifest

In [ ]:
manifest_path = Path("../data/raw/clinicaltrials_project_dev_200_manifest.json")

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Manifest saved:", manifest_path)
print("File exists:", manifest_path.exists())

In [ ]:
# Build trial-level table from 200-study extract

trial_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    design = protocol.get("designModule", {})

    trial_rows.append({
        "nct_id": identification.get("nctId"),
        "brief_title": identification.get("briefTitle"),
        "overall_status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "completion_date": status.get("completionDateStruct", {}).get("date"),
        "lead_sponsor": sponsor.get("leadSponsor", {}).get("name"),
        "sponsor_class": sponsor.get("leadSponsor", {}).get("class"),
        "study_type": design.get("studyType"),
        "phases": design.get("phases"),
        "enrollment": design.get("enrollmentInfo", {}).get("count")
    })

trials_df = pd.DataFrame(trial_rows)

trials_df.head()

In [ ]:
print("Rows:", len(trials_df))
print("Unique NCT IDs:", trials_df["nct_id"].nunique())
print("Duplicate NCT IDs:", trials_df["nct_id"].duplicated().sum())

trials_df.info()

In [ ]:
# Build interventions child table from 200-study extract

intervention_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    arms_interventions = protocol.get("armsInterventionsModule", {})

    nct_id = identification.get("nctId")
    interventions = arms_interventions.get("interventions", [])

    for intervention in interventions:
        intervention_rows.append({
            "nct_id": nct_id,
            "intervention_type": intervention.get("type"),
            "intervention_name": intervention.get("name"),
            "description": intervention.get("description")
        })

interventions_df = pd.DataFrame(intervention_rows)

interventions_df.head()

In [ ]:
print("Intervention rows:", len(interventions_df))
print("Studies represented:", interventions_df["nct_id"].nunique())

invalid_intervention_keys = interventions_df[
    ~interventions_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_intervention_keys))

interventions_df.info()

In [ ]:
# Build conditions child table from 200-study extract

condition_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    conditions_module = protocol.get("conditionsModule", {})

    nct_id = identification.get("nctId")
    conditions = conditions_module.get("conditions", [])

    for condition in conditions:
        condition_rows.append({
            "nct_id": nct_id,
            "condition": condition
        })

conditions_df = pd.DataFrame(condition_rows)

conditions_df.head()

In [ ]:
print("Condition rows:", len(conditions_df))
print("Studies represented:", conditions_df["nct_id"].nunique())

invalid_condition_keys = conditions_df[
    ~conditions_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_condition_keys))

conditions_df.info()

In [ ]:
# Build locations child table from 200-study extract

location_rows = []

for study in all_studies:

    protocol = study.get("protocolSection", {})
    identification = protocol.get("identificationModule", {})
    contacts_locations = protocol.get("contactsLocationsModule", {})

    nct_id = identification.get("nctId")
    locations = contacts_locations.get("locations", [])

    for location in locations:
        location_rows.append({
            "nct_id": nct_id,
            "facility": location.get("facility"),
            "city": location.get("city"),
            "state": location.get("state"),
            "country": location.get("country")
        })

locations_df = pd.DataFrame(location_rows)

locations_df.head()

In [ ]:
print("Location rows:", len(locations_df))
print("Studies represented:", locations_df["nct_id"].nunique())

invalid_location_keys = locations_df[
    ~locations_df["nct_id"].isin(trials_df["nct_id"])
]

print("Invalid foreign keys:", len(invalid_location_keys))

locations_df.info()

In [ ]:
# Clean trial-level date fields

trials_df["start_date"] = pd.to_datetime(
    trials_df["start_date"],
    errors="coerce",
    format="mixed"
)

trials_df["completion_date"] = pd.to_datetime(
    trials_df["completion_date"],
    errors="coerce",
    format="mixed"
)

print(trials_df[["start_date", "completion_date"]].dtypes)

In [ ]:
# Study type and sponsor coverage

print("Study types:")
print(trials_df["study_type"].value_counts(dropna=False))

print("\nSponsor classes:")
print(trials_df["sponsor_class"].value_counts(dropna=False))

print("\nOverall status:")
print(trials_df["overall_status"].value_counts(dropna=False))

In [ ]:
# Intervention type coverage

print(
    interventions_df["intervention_type"]
    .value_counts(dropna=False)
)
# Date coverage

print("Earliest start:", trials_df["start_date"].min())
print("Latest start:", trials_df["start_date"].max())

In [ ]:
# Final structural assertions

assert len(trials_df) == 200
assert trials_df["nct_id"].is_unique
assert trials_df["nct_id"].notna().all()

assert interventions_df["nct_id"].isin(trials_df["nct_id"]).all()
assert conditions_df["nct_id"].isin(trials_df["nct_id"]).all()
assert locations_df["nct_id"].isin(trials_df["nct_id"]).all()

print("All structural QA checks passed.")

In [ ]:
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Save processed analytical tables

processed_path = Path("../data/processed")

trials_df.to_csv(
    processed_path / "trials_dev_200.csv",
    index=False
)

interventions_df.to_csv(
    processed_path / "interventions_dev_200.csv",
    index=False
)

conditions_df.to_csv(
    processed_path / "conditions_dev_200.csv",
    index=False
)

locations_df.to_csv(
    processed_path / "locations_dev_200.csv",
    index=False
)

print("Processed tables saved.")

In [ ]:
for file in processed_path.glob("*_dev_200.csv"):
    print(file.name, "-", file.exists())

In [ ]:
# Ingestion validation summary

print("=== INGESTION SUMMARY ===")

print("\nRaw extraction:")
print("Studies downloaded:", len(all_studies))
print("Pages downloaded:", pages_downloaded)

print("\nTrial table:")
print("Rows:", len(trials_df))
print("Unique NCT IDs:", trials_df["nct_id"].nunique())
print("Duplicate NCT IDs:", trials_df["nct_id"].duplicated().sum())

print("\nChild tables:")
print("Intervention rows:", len(interventions_df))
print("Condition rows:", len(conditions_df))
print("Location rows:", len(locations_df))

print("\nForeign key validation:")
print(
    "Interventions:",
    interventions_df["nct_id"].isin(trials_df["nct_id"]).all()
)
print(
    "Conditions:",
    conditions_df["nct_id"].isin(trials_df["nct_id"]).all()
)
print(
    "Locations:",
    locations_df["nct_id"].isin(trials_df["nct_id"]).all()
)

print("\nINGESTION PIPELINE COMPLETE")